In [7]:
import cv2
import numpy as np
import tensorflow as tf

print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)
print("TensorFlow:", tf.__version__)

import os
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout


OpenCV: 4.12.0
NumPy: 1.23.5
TensorFlow: 2.12.0


In [8]:
import numpy as np
import tensorflow as tf
import cv2

print("NumPy:", np.__version__)
print("TF:", tf.__version__)
print("OpenCV:", cv2.__version__)


NumPy: 1.23.5
TF: 2.12.0
OpenCV: 4.12.0


In [9]:
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10

TRAIN_DIR = "dataset/train"
TEST_DIR = "dataset/test"

CLASSES = ["ORGANIC", "RECYCLABLE"]


In [10]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_data = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)


Found 22564 images belonging to 2 classes.
Found 2513 images belonging to 2 classes.


In [11]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(2, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 222, 222, 32)      896       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 111, 111, 32)     0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 109, 109, 64)      18496     
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 54, 54, 64)       0         
 2D)                                                             
                                                                 
 conv2d_2 (Conv2D)           (None, 52, 52, 128)       73856     
                                                                 
 max_pooling2d_2 (MaxPooling  (None, 26, 26, 128)      0

In [5]:
history = model.fit(
    train_data,
    epochs=EPOCHS,
    validation_data=test_data
)


NameError: name 'model' is not defined

In [12]:
model.save("waste_classifier.h5")
print("Model saved!")


Model saved!


In [11]:
import cv2
import numpy as np

img = cv2.imread("DATASET\TEST\R\R_10926.jpg")  # put a test image
img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
img = img / 255.0
img = np.reshape(img, (1, IMG_SIZE, IMG_SIZE, 3))

prediction = model.predict(img)
print(CLASSES[np.argmax(prediction)])


1/1 [==============================] - 0s 38ms/step
RECYCLABLE


In [14]:
CONF_THRESHOLD = 0.4
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    img = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    img = np.reshape(img, (1, IMG_SIZE, IMG_SIZE, 3))

    prediction = model.predict(img, verbose=0)[0]
    confidence = np.max(prediction)
    class_id = np.argmax(prediction)

    if confidence < CONF_THRESHOLD:
        label = "No Object"
    else:
        label = CLASSES[class_id]

    cv2.putText(
        frame,
        f"{label} ({confidence:.2f})",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    cv2.imshow("Waste Classifier", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
